# Explainable BrainNetCNN with Cross-Modal Contrastive Phenotypic Alignment (ABIDE I)
### Standalone Kaggle Training Notebook for 2x Tesla T4 GPUs with Stratified 10-Fold CV

This notebook provides a completely self-contained, reproducible training and evaluation pipeline for Autism Spectrum Disorder (ASD) classification on the ABIDE I consortium dataset (Craddock CC200 parcellation).

#### Key Components
- **Dataset Auto-Discovery**: Automatic indexing of local Kaggle dataset paths (`/kaggle/input/`) with automated fallback to Nilearn ABIDE PCP CPAC fetching.
- **Quality Control Scrubbing**: Motion filtering retaining subjects with `func_mean_fd <= 0.5 mm`.
- **Leak-Free Tangent Space FC**: Riemannian mean covariance estimated strictly on training folds and projected to test splits.
- **Clinical Phenotypic Prompt Formulation**: Natural language clinical profiles excluding diagnostic labels.
- **BrainNetCNN Connectome Encoder**: Topological E2E cross-convolutions, E2N node pooling, and N2G graph summarization.
- **Dual-Modal InfoNCE Alignment**: Symmetric contrastive learning aligning connectome and clinical text representations with differentiable transformer backbone.
- **2x Tesla T4 Multi-GPU Training**: Native DataParallel parallelization with FP16 Automatic Mixed Precision (`torch.amp.autocast`).
- **Stratified 10-Fold Cross-Validation**: Leak-free evaluation and out-of-fold prediction aggregation.
- **Publication-Grade Evaluation**: Aggregate ROC curve with ROC-AUC, Balanced Accuracy, Sensitivity, Specificity, and Macro-F1.
- **Neurobiological Explainability**: CC200 to Yeo 7 network mapping, Grad-CAM attribution on `E2N`, Network Enrichment Scores ($E_{\text{net}}$), and Quantitative Fidelity perturbation tests.

#### Kaggle CLI Remote Workflow
```bash
# 1. Push notebook to Kaggle
kaggle kernels push -p notebooks

# 2. Check kernel run status
kaggle kernels status kiettuan/05-explainable-brainnet-clip-training

# 3. Retrieve trained model checkpoints and publication figures
kaggle kernels output kiettuan/05-explainable-brainnet-clip-training -p artifacts/kaggle_outputs
```

In [ ]:
# Environment and dependency verification
import subprocess
import sys

def install_if_missing(package_name: str, import_name: str | None = None) -> None:
    if import_name is None:
        import_name = package_name
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing {package_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

install_if_missing("nilearn")
install_if_missing("transformers")
install_if_missing("scikit-learn", "sklearn")
install_if_missing("pandas")
install_if_missing("numpy")
install_if_missing("matplotlib")
print("SUCCESS: Environment dependencies verified successfully.")

In [ ]:
import os
import glob
import json
import math
import random
import copy
import time
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    roc_auc_score,
    roc_curve,
    auc,
)
from sklearn.model_selection import StratifiedKFold
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from nilearn import datasets
from nilearn.connectome import ConnectivityMeasure

# Reproducibility seed setup
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Real-time progress logger utility
def log_msg(msg: str, log_file: Optional[str] = "training_progress.log") -> None:
    """Log message with formatted timestamp to stdout and optional disk file."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    formatted = f"[{timestamp}] {msg}"
    print(formatted, flush=True)
    if log_file:
        try:
            with open(log_file, "a", encoding="utf-8") as f:
                f.write(formatted + "\n")
        except Exception:
            pass

# Device configuration and multi-GPU detection (2x Tesla T4)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
log_msg(f"[INFO] Execution Device: {DEVICE}")
log_msg(f"[INFO] Available GPU count: {NUM_GPUS}")
if NUM_GPUS > 1:
    for i in range(NUM_GPUS):
        log_msg(f"  [GPU {i}]: {torch.cuda.get_device_name(i)}")
    log_msg("[INFO] Multi-GPU DataParallel parallelization active (accelerator: nvidiaTeslaT4x2).")

In [ ]:
def discover_or_fetch_abide_dataset(
    cache_dir: str = "./abide_data",
) -> Tuple[pd.DataFrame, List[np.ndarray], List[str]]:
    """
    Auto-discover ABIDE CC200 files in Kaggle input paths or download via Nilearn.

    Returns:
        pheno_df: Phenotypic DataFrame with subject demographics and clinical scores.
        timeseries_list: List of ROI BOLD time-series arrays of shape (T, 200).
        subject_ids: List of subject ID strings.
    """
    kaggle_input_paths = [
        "/kaggle/input/abide-pcp-cc200",
        "/kaggle/input/abide-i-dataset",
        "/kaggle/input/abide1",
        "./abide_data",
        cache_dir,
    ]

    found_pheno = None
    ts_dir = None

    for base_dir in kaggle_input_paths:
        p = Path(base_dir)
        if not p.exists():
            continue
        pheno_candidates = list(p.glob("**/Phenotypic_V1_0b_preprocessed1.csv")) + list(p.glob("**/*phenotypic*.csv"))
        if pheno_candidates:
            found_pheno = pheno_candidates[0]
            ts_dir = found_pheno.parent
            print(f"Discovered local dataset at: {found_pheno.parent}")
            break

    if found_pheno is not None and ts_dir is not None:
        pheno_df = pd.read_csv(found_pheno)
        ts_files = sorted(list(ts_dir.glob("**/*_rois_cc200.1D")) + list(ts_dir.glob("**/*.1D")))
        if len(ts_files) > 0:
            print(f"Found {len(ts_files)} timeseries files locally.")
            ts_dict: Dict[int, np.ndarray] = {}
            for f in ts_files:
                parts = f.stem.split("_")
                for sub_id_token in parts:
                    if sub_id_token.isdigit() and len(sub_id_token) >= 4:
                        try:
                            ts_data = np.loadtxt(f)
                            if ts_data.ndim == 2 and ts_data.shape[1] == 200:
                                ts_dict[int(sub_id_token)] = ts_data.astype(np.float32)
                        except Exception:
                            pass
                        break

            matched_indices = []
            matched_ts = []
            for idx, row in pheno_df.iterrows():
                sub_id = int(row["SUB_ID"]) if "SUB_ID" in row and pd.notna(row["SUB_ID"]) else -1
                if sub_id in ts_dict:
                    matched_indices.append(idx)
                    matched_ts.append(ts_dict[sub_id])

            if len(matched_ts) > 50:
                filtered_df = pheno_df.iloc[matched_indices].reset_index(drop=True)
                sub_ids = [str(x) for x in filtered_df["SUB_ID"]]
                print(f"Successfully indexed {len(matched_ts)} matched subjects from local Kaggle inputs.")
                return filtered_df, matched_ts, sub_ids

    # Remote download via Nilearn ABIDE PCP CPAC
    print("INFO: Fetching ABIDE PCP CC200 data via Nilearn (CPAC pipeline)...\nThis may take several minutes.")
    dataset = datasets.fetch_abide_pcp(
        pipeline="cpac",
        derivatives=["rois_cc200"],
        quality_checked=False,
        data_dir=cache_dir,
    )
    pheno_df = pd.DataFrame(dataset.phenotypic)
    ts_list = [np.asarray(ts, dtype=np.float32) for ts in dataset.rois_cc200]
    sub_ids = [str(sid) for sid in pheno_df["SUB_ID"]]
    print(f"SUCCESS: Fetched {len(ts_list)} subjects via Nilearn ABIDE PCP.")
    return pheno_df, ts_list, sub_ids

In [ ]:
def filter_motion_quality(
    pheno_df: pd.DataFrame,
    timeseries_list: List[np.ndarray],
    max_fd: float = 0.5,
) -> Tuple[pd.DataFrame, List[np.ndarray], np.ndarray]:
    """Filter subjects exceeding Mean Framewise Displacement (FD > 0.5 mm)."""
    if "func_mean_fd" not in pheno_df.columns:
        warnings.warn("Column 'func_mean_fd' not found. Skipping motion filtering.")
        mask = np.ones(len(pheno_df), dtype=bool)
        return pheno_df.copy(), timeseries_list, mask

    fd = pd.to_numeric(pheno_df["func_mean_fd"], errors="coerce")
    keep_mask = fd.notna() & (fd >= 0.0) & (fd <= max_fd)
    filtered_df = pheno_df[keep_mask].copy().reset_index(drop=True)
    filtered_ts = [timeseries_list[i] for i, k in enumerate(keep_mask) if k]
    print(f"Motion filtering (FD <= {max_fd}mm): retained {len(filtered_df)} / {len(pheno_df)} subjects.")
    return filtered_df, filtered_ts, keep_mask.to_numpy()


def generate_phenotypic_prompts(pheno_df: pd.DataFrame) -> List[str]:
    """
    Format clinical text prompts without diagnostic label leakage.

    Format:
    'A {age:.1f}-year-old {sex_str} individual, Full-scale IQ {fiq_str},
     Verbal IQ {viq_str}, evaluated on ADOS with Social-Affect score {ados_soc_str}
     and Repetitive Behavior score {ados_rbr_str}.'
    """
    prompts = []
    for _, row in pheno_df.iterrows():
        age_val = row.get("AGE_AT_SCAN", np.nan)
        try:
            age_num = float(age_val)
            age_str = f"{age_num:.1f}" if not np.isnan(age_num) else "not documented"
        except (ValueError, TypeError):
            age_str = "not documented"

        sex_val = row.get("SEX", np.nan)
        try:
            sex_int = int(float(sex_val))
            sex_str = "male" if sex_int == 1 else ("female" if sex_int == 2 else "individual")
        except (ValueError, TypeError):
            sex_str = "individual"

        def _fmt_score(val: Any) -> str:
            try:
                num = float(val)
                return f"{num:.1f}" if not np.isnan(num) and num >= 0 else "not documented"
            except (ValueError, TypeError):
                return "not documented"

        fiq_str = _fmt_score(row.get("FIQ", np.nan))
        viq_str = _fmt_score(row.get("VIQ", np.nan))
        ados_soc_str = _fmt_score(row.get("ADOS_GOTHAM_SOCAFFECT", row.get("ADOS_TOTAL", np.nan)))
        ados_rbr_str = _fmt_score(row.get("ADOS_GOTHAM_RRB", np.nan))

        prompt = (
            f"A {age_str}-year-old {sex_str} individual, Full-scale IQ {fiq_str}, "
            f"Verbal IQ {viq_str}, evaluated on ADOS with Social-Affect score {ados_soc_str} "
            f"and Repetitive Behavior score {ados_rbr_str}."
        )
        prompts.append(prompt)
    return prompts

In [ ]:
def compute_tangent_fc(
    time_series_list: List[np.ndarray],
    fit_indices: Optional[Sequence[int]] = None,
) -> np.ndarray:
    """
    Compute symmetric Riemannian Tangent Space Functional Connectivity matrices.
    Reference Riemannian covariance is fit strictly on training split subjects.
    """
    measure = ConnectivityMeasure(kind="tangent", vectorize=False, discard_diagonal=False)
    if fit_indices is not None:
        fit_data = [time_series_list[i] for i in fit_indices]
        measure.fit(fit_data)
        fc_matrices = measure.transform(time_series_list)
    else:
        fc_matrices = measure.fit_transform(time_series_list)
    return np.asarray(fc_matrices, dtype=np.float32)

In [ ]:
class BrainPhenoDataset(Dataset):
    """Multi-modal dataset pairing CC200 FC matrices with phenotypic text prompts."""
    def __init__(
        self,
        fc_matrices: np.ndarray,
        prompts: List[str],
        labels: np.ndarray,
        site_ids: Optional[Sequence[str]] = None,
    ):
        self.fc_matrices = torch.from_numpy(fc_matrices).float()
        self.prompts = list(prompts)
        self.labels = torch.from_numpy(labels).long()
        self.site_ids = list(site_ids) if site_ids is not None else ["" for _ in range(len(labels))]

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return {
            "fc": self.fc_matrices[idx],
            "prompt": self.prompts[idx],
            "label": self.labels[idx],
            "site_id": self.site_ids[idx],
        }


def collate_brain_pheno(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "fc": torch.stack([item["fc"] for item in batch], dim=0),
        "prompts": [item["prompt"] for item in batch],
        "labels": torch.stack([item["label"] for item in batch], dim=0),
        "site_ids": [item["site_id"] for item in batch],
    }

In [ ]:
class E2EBlock(nn.Module):
    """Edge-to-Edge convolutional block with horizontal and vertical cross-filters."""
    def __init__(self, num_rois: int = 200, in_channels: int = 1, out_channels: int = 32, dropout: float = 0.1):
        super().__init__()
        self.conv_h = nn.Conv2d(in_channels, out_channels, kernel_size=(1, num_rois), bias=True)
        self.conv_v = nn.Conv2d(in_channels, out_channels, kernel_size=(num_rois, 1), bias=True)
        self.bn = nn.BatchNorm2d(out_channels)
        self.activation = nn.LeakyReLU(negative_slope=0.33, inplace=True)
        self.dropout = nn.Dropout(p=dropout) if dropout > 0.0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.conv_h(x)
        v = self.conv_v(x)
        out = h.expand(-1, -1, -1, x.size(-1)) + v.expand(-1, -1, x.size(-2), -1)
        out = self.bn(out)
        out = self.activation(out)
        out = self.dropout(out)
        return out


class E2NBlock(nn.Module):
    """Edge-to-Node convolutional block pooling incident connections per region."""
    def __init__(self, num_rois: int = 200, in_channels: int = 32, out_channels: int = 64, dropout: float = 0.1):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=(1, num_rois), bias=True)
        self.bn = nn.BatchNorm1d(out_channels)
        self.activation = nn.LeakyReLU(negative_slope=0.33, inplace=True)
        self.dropout = nn.Dropout(p=dropout) if dropout > 0.0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.conv(x).squeeze(-1)
        out = self.bn(out)
        out = self.activation(out)
        out = self.dropout(out)
        return out


class N2GBlock(nn.Module):
    """Node-to-Graph convolutional block aggregating across all brain regions."""
    def __init__(self, num_rois: int = 200, in_channels: int = 64, out_channels: int = 128):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=num_rois, bias=True)
        self.bn = nn.BatchNorm1d(out_channels)
        self.activation = nn.LeakyReLU(negative_slope=0.33, inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.conv(x)
        if self.training and out.size(0) == 1:
            out = self.activation(out)
        else:
            out = self.bn(out)
            out = self.activation(out)
        return out


class BrainNetCNN(nn.Module):
    """BrainNetCNN Connectome Encoder with projection head and classifier."""
    def __init__(
        self,
        num_rois: int = 200,
        in_channels: int = 1,
        e2e_channels: int = 32,
        e2n_channels: int = 64,
        n2g_channels: int = 128,
        embed_dim: int = 128,
        num_classes: int = 2,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.e2e = E2EBlock(num_rois, in_channels, e2e_channels, dropout)
        self.e2n = E2NBlock(num_rois, e2e_channels, e2n_channels, dropout)
        self.n2g = N2GBlock(num_rois, e2n_channels, n2g_channels)
        self.proj = nn.Linear(n2g_channels, embed_dim)
        self.classifier = nn.Sequential(
            nn.Linear(n2g_channels, 64),
            nn.LeakyReLU(0.33, inplace=True),
            nn.Dropout(p=dropout),
            nn.Linear(64, num_classes),
        )

    def forward(
        self,
        x: torch.Tensor,
        return_embedding: bool = False,
    ) -> Union[torch.Tensor, Tuple[torch.Tensor, torch.Tensor]]:
        if x.dim() == 3:
            x = x.unsqueeze(1)
        e2e_out = self.e2e(x)
        e2n_out = self.e2n(e2e_out)
        n2g_out = self.n2g(e2n_out)
        graph_feat = n2g_out.squeeze(-1)
        logits = self.classifier(graph_feat)

        if return_embedding:
            z_brain = self.proj(graph_feat)
            z_brain = F.normalize(z_brain, p=2, dim=-1)
            return logits, z_brain
        return logits

In [ ]:
class PhenotypicTextEncoder(nn.Module):
    """
    Clinical text encoder projecting phenotypic descriptions to shared latent space.
    Uses HuggingFace AutoModel / AutoTokenizer with frozen early layers and differentiable forward.
    """
    def __init__(
        self,
        model_name: str = "sentence-transformers/all-MiniLM-L6-v2",
        embed_dim: int = 128,
        freeze_early_layers: bool = True,
        n_finetune_layers: int = 2,
    ):
        super().__init__()
        self.model_name = model_name
        self.embed_dim = embed_dim
        self.has_transformers = False

        try:
            from transformers import AutoModel, AutoTokenizer
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.transformer = AutoModel.from_pretrained(model_name)
            hidden_dim = self.transformer.config.hidden_size

            if freeze_early_layers:
                if hasattr(self.transformer, "embeddings"):
                    for param in self.transformer.embeddings.parameters():
                        param.requires_grad = False
                if hasattr(self.transformer, "encoder") and hasattr(self.transformer.encoder, "layer"):
                    layers = self.transformer.encoder.layer
                    freeze_until = max(0, len(layers) - n_finetune_layers)
                    for i in range(freeze_until):
                        for param in layers[i].parameters():
                            param.requires_grad = False
            self.has_transformers = True
        except Exception as e:
            print(f"INFO: Transformers backbone unavailable ({e}), using lightweight fallback text encoder.")
            self.tokenizer = None
            self.transformer = None
            hidden_dim = 256

        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, embed_dim),
            nn.LayerNorm(embed_dim),
        )

    def forward(self, texts: List[str], device: torch.device) -> torch.Tensor:
        if self.has_transformers and self.transformer is not None and self.tokenizer is not None:
            tokens = self.tokenizer(
                texts,
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt",
            ).to(device)
            outputs = self.transformer(**tokens)
            token_embeddings = outputs.last_hidden_state
            input_mask = tokens["attention_mask"].unsqueeze(-1).expand(token_embeddings.size()).float()
            sum_embeds = torch.sum(token_embeddings * input_mask, dim=1)
            sum_mask = torch.clamp(input_mask.sum(dim=1), min=1e-9)
            pooled = sum_embeds / sum_mask
            proj = self.projection(pooled)
        else:
            batch_size = len(texts)
            fake_embeds = torch.zeros((batch_size, 256), device=device)
            for i, txt in enumerate(texts):
                vals = [ord(c) % 256 for c in txt[:256]]
                if vals:
                    fake_embeds[i, :len(vals)] = torch.tensor(vals, dtype=torch.float32, device=device) / 256.0
            proj = self.projection(fake_embeds)

        return F.normalize(proj, p=2, dim=-1)

In [ ]:
class DualModalLoss(nn.Module):
    """
    Joint multi-task loss combining Label-Smoothed Cross-Entropy with symmetric InfoNCE.
    L_total = L_CE + lambda * L_InfoNCE
    """
    def __init__(self, lambda_align: float = 0.3, label_smoothing: float = 0.1, temperature: float = 0.07):
        super().__init__()
        self.lambda_align = lambda_align
        self.ce_loss = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
        self.log_tau = nn.Parameter(torch.log(torch.tensor(temperature, dtype=torch.float32)))

    @property
    def temperature(self) -> torch.Tensor:
        return torch.exp(self.log_tau).clamp(min=1e-3, max=1.0)

    def infonce_loss(self, z_brain: torch.Tensor, z_text: torch.Tensor) -> torch.Tensor:
        tau = self.temperature
        sim = torch.matmul(z_brain, z_text.t()) / tau
        targets = torch.arange(sim.size(0), device=sim.device, dtype=torch.long)
        loss_b2t = F.cross_entropy(sim, targets)
        loss_t2b = F.cross_entropy(sim.t(), targets)
        return 0.5 * (loss_b2t + loss_t2b)

    def forward(
        self,
        logits: torch.Tensor,
        targets: torch.Tensor,
        z_brain: Optional[torch.Tensor] = None,
        z_text: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        l_ce = self.ce_loss(logits, targets)
        if z_brain is not None and z_text is not None and self.lambda_align > 0.0:
            l_align = self.infonce_loss(z_brain, z_text)
            total = l_ce + self.lambda_align * l_align
        else:
            l_align = torch.tensor(0.0, device=logits.device)
            total = l_ce
        return total, l_ce, l_align

In [ ]:
def train_one_epoch(
    brain_net: nn.Module,
    text_encoder: nn.Module,
    criterion: DualModalLoss,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: Optional[Any],
    device: torch.device,
    use_amp: bool = True,
) -> Tuple[float, float, float]:
    brain_net.train()
    text_encoder.train()
    total_loss, total_ce, total_align = 0.0, 0.0, 0.0

    for batch in dataloader:
        optimizer.zero_grad()
        fc = batch["fc"].to(device)
        labels = batch["labels"].to(device)
        prompts = batch["prompts"]

        with torch.amp.autocast("cuda", enabled=use_amp and device.type == "cuda"):
            # Invoke DataParallel directly to enable multi-GPU scatter across 2x Tesla T4
            logits, z_brain = brain_net(fc, return_embedding=True)
            z_text = text_encoder(prompts, device=device)
            loss, ce, align = criterion(logits, labels, z_brain, z_text)

        if scaler is not None and use_amp and device.type == "cuda":
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(brain_net.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(brain_net.parameters(), max_norm=1.0)
            optimizer.step()

        total_loss += loss.item()
        total_ce += ce.item()
        total_align += align.item()

    n = max(1, len(dataloader))
    return total_loss / n, total_ce / n, total_align / n


@torch.no_grad()
def evaluate_unimodal(
    brain_net: nn.Module,
    dataloader: DataLoader,
    device: torch.device,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Unimodal evaluation evaluating strictly resting-state fMRI matrices without text."""
    brain_net.eval()
    all_targets = []
    all_probs = []
    all_preds = []

    for batch in dataloader:
        fc = batch["fc"].to(device)
        labels = batch["labels"].cpu().numpy()
        logits = brain_net(fc, return_embedding=False)
        probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        preds = torch.argmax(logits, dim=-1).cpu().numpy()

        all_targets.append(labels)
        all_probs.append(probs)
        all_preds.append(preds)

    return (
        np.concatenate(all_targets, axis=0),
        np.concatenate(all_probs, axis=0),
        np.concatenate(all_preds, axis=0),
    )

In [ ]:
def run_stratified_10_fold_cv(
    timeseries_list: List[np.ndarray],
    prompts: List[str],
    labels: np.ndarray,
    site_ids: Sequence[str],
    n_splits: int = 10,
    epochs: int = 50,
    batch_size: int = 32,
    lr_brain: float = 1e-3,
    lr_text: float = 1e-5,
    lambda_align: float = 0.3,
    device: torch.device = DEVICE,
) -> Tuple[Dict[str, Any], BrainNetCNN]:
    """
    Executes leak-free Stratified 10-Fold Cross Validation with real-time fold logging.
    """
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof_targets = np.zeros(len(labels), dtype=np.int64)
    oof_probs = np.zeros(len(labels), dtype=np.float32)
    oof_preds = np.zeros(len(labels), dtype=np.int64)
    fold_metrics = []

    best_overall_auc = -1.0
    best_overall_state_dict: Optional[Dict[str, Any]] = None

    use_amp = (device.type == "cuda")
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    else:
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    cv_start_time = time.time()
    log_msg(f"[INFO] Starting Stratified {n_splits}-Fold CV across {len(labels)} subjects ({np.sum(labels == 1)} ASD, {np.sum(labels == 0)} TC).")

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(timeseries_list, labels)):
        fold_start_time = time.time()
        train_asd = int(np.sum(labels[train_idx] == 1))
        train_tc = int(np.sum(labels[train_idx] == 0))
        val_asd = int(np.sum(labels[val_idx] == 1))
        val_tc = int(np.sum(labels[val_idx] == 0))

        log_msg(f"\n[INFO] ==================== FOLD {fold_idx + 1:02d} / {n_splits:02d} START ====================")
        log_msg(f"[INFO] [Fold {fold_idx + 1:02d}/{n_splits:02d}] Split partition: Train={len(train_idx)} (ASD={train_asd}, TC={train_tc}) | Val={len(val_idx)} (ASD={val_asd}, TC={val_tc})")
        
        # Leak-free Tangent space fitting strictly on training split
        t_fc_start = time.time()
        fc_train = compute_tangent_fc(timeseries_list, fit_indices=train_idx)
        train_matrices = fc_train[train_idx]
        val_matrices = fc_train[val_idx]
        log_msg(f"[INFO] [Fold {fold_idx + 1:02d}/{n_splits:02d}] Tangent FC covariance fit completed in {time.time() - t_fc_start:.2f}s.")

        train_ds = BrainPhenoDataset(
            train_matrices,
            [prompts[i] for i in train_idx],
            labels[train_idx],
            [site_ids[i] for i in train_idx],
        )
        val_ds = BrainPhenoDataset(
            val_matrices,
            [prompts[i] for i in val_idx],
            labels[val_idx],
            [site_ids[i] for i in val_idx],
        )

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, collate_fn=collate_brain_pheno)
        val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, collate_fn=collate_brain_pheno)

        brain_net = BrainNetCNN(num_rois=200, embed_dim=128, num_classes=2).to(device)
        text_encoder = PhenotypicTextEncoder(embed_dim=128).to(device)
        criterion = DualModalLoss(lambda_align=lambda_align).to(device)

        if torch.cuda.device_count() > 1 and device.type == "cuda":
            brain_net_parallel: nn.Module = nn.DataParallel(brain_net)
        else:
            brain_net_parallel = brain_net

        optimizer = torch.optim.AdamW([
            {"params": brain_net.parameters(), "lr": lr_brain, "weight_decay": 1e-4},
            {"params": text_encoder.parameters(), "lr": lr_text, "weight_decay": 1e-4},
            {"params": [criterion.log_tau], "lr": lr_text},
        ])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

        best_fold_auc = 0.0
        best_fold_epoch = 0
        best_fold_probs = None
        best_fold_preds = None

        for epoch in range(1, epochs + 1):
            ep_start = time.time()
            loss, ce, align = train_one_epoch(
                brain_net_parallel, text_encoder, criterion, train_loader, optimizer, scaler, device, use_amp
            )
            scheduler.step()
            ep_duration = time.time() - ep_start

            if epoch % 5 == 0 or epoch == 1 or epoch == epochs:
                y_true, y_prob, y_pred = evaluate_unimodal(brain_net_parallel, val_loader, device)
                auc_score = float(roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else 0.5
                acc = float(accuracy_score(y_true, y_pred))
                bacc = float(balanced_accuracy_score(y_true, y_pred))
                current_tau = float(criterion.temperature.item())

                log_msg(
                    f"  [Fold {fold_idx + 1:02d}/{n_splits:02d}] Epoch {epoch:02d}/{epochs:02d} | "
                    f"Train Loss: {loss:.4f} (CE: {ce:.4f}, Align: {align:.4f}, Tau: {current_tau:.3f}) | "
                    f"Val AUC: {auc_score:.4f} | BACC: {bacc:.4f} | ACC: {acc:.4f} ({ep_duration:.1f}s)"
                )

                if auc_score >= best_fold_auc:
                    best_fold_auc = auc_score
                    best_fold_epoch = epoch
                    best_fold_probs = y_prob
                    best_fold_preds = y_pred
                    if auc_score > best_overall_auc:
                        best_overall_auc = auc_score
                        raw_model = brain_net_parallel.module if isinstance(brain_net_parallel, nn.DataParallel) else brain_net_parallel
                        best_overall_state_dict = copy.deepcopy(raw_model.state_dict())

        oof_targets[val_idx] = labels[val_idx]
        oof_probs[val_idx] = best_fold_probs if best_fold_probs is not None else y_prob
        oof_preds[val_idx] = best_fold_preds if best_fold_preds is not None else y_pred

        fold_duration = time.time() - fold_start_time
        final_fold_acc = float(accuracy_score(labels[val_idx], oof_preds[val_idx]))
        final_fold_bacc = float(balanced_accuracy_score(labels[val_idx], oof_preds[val_idx]))

        fold_metrics.append({
            "fold": fold_idx + 1,
            "auc": float(best_fold_auc),
            "acc": final_fold_acc,
            "bacc": final_fold_bacc,
            "duration_seconds": float(np.round(fold_duration, 2)),
        })

        running_aucs = [m["auc"] for m in fold_metrics]
        log_msg(
            f"[SUCCESS] [Fold {fold_idx + 1:02d}/{n_splits:02d}] Completed in {fold_duration:.1f}s | "
            f"Best Val AUC: {best_fold_auc:.4f} (Epoch {best_fold_epoch:02d}) | "
            f"BACC: {final_fold_bacc:.4f} | ACC: {final_fold_acc:.4f}"
        )
        log_msg(
            f"[INFO] Cross-Validation Progress: {fold_idx + 1}/{n_splits} folds complete | "
            f"Running Mean AUC: {np.mean(running_aucs):.4f} +/- {np.std(running_aucs):.4f}"
        )

    total_cv_time = time.time() - cv_start_time
    log_msg(f"\n[SUCCESS] All {n_splits} folds completed in {total_cv_time / 60.0:.2f} minutes.")

    cv_results = {
        "fold_metrics": fold_metrics,
        "oof_targets": oof_targets,
        "oof_probs": oof_probs,
        "oof_preds": oof_preds,
    }
    # Reconstruct pristine model instance loaded with the highest-validation state dict
    final_best_model = BrainNetCNN(num_rois=200, embed_dim=128, num_classes=2).to(device)
    if best_overall_state_dict is not None:
        final_best_model.load_state_dict(best_overall_state_dict)
    return cv_results, final_best_model

In [ ]:
def compute_and_plot_aggregate_metrics(
    oof_targets: np.ndarray,
    oof_probs: np.ndarray,
    oof_preds: np.ndarray,
    output_fig_path: str = "aggregate_roc_curve.png",
    output_csv_path: str = "oof_predictions.csv",
) -> Dict[str, float]:
    """Compute aggregate CV metrics and plot aggregate ROC curve."""
    acc = float(accuracy_score(oof_targets, oof_preds))
    bacc = float(balanced_accuracy_score(oof_targets, oof_preds))
    roc_auc = float(roc_auc_score(oof_targets, oof_probs))
    f1 = float(f1_score(oof_targets, oof_preds, average="macro"))

    tn, fp, fn, tp = confusion_matrix(oof_targets, oof_preds).ravel()
    sensitivity = float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0
    specificity = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0

    metrics = {
        "Accuracy": acc,
        "Balanced Accuracy": bacc,
        "Sensitivity (Recall)": sensitivity,
        "Specificity": specificity,
        "ROC-AUC": roc_auc,
        "Macro-F1": f1,
    }

    print("\n" + "=" * 50)
    print("AGGREGATE OUT-OF-FOLD CROSS-VALIDATION RESULTS")
    print("=" * 50)
    for k, v in metrics.items():
        print(f"  {k:24s}: {v:.4f}")
    print("=" * 50)

    df_oof = pd.DataFrame({
        "y_true": oof_targets,
        "y_prob": oof_probs,
        "y_pred": oof_preds,
    })
    df_oof.to_csv(output_csv_path, index=False)
    print(f"Saved out-of-fold predictions to: {output_csv_path}")

    fpr, tpr, _ = roc_curve(oof_targets, oof_probs)
    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, color="#1f77b4", lw=2.5, label=f"OOF ROC Curve (AUC = {roc_auc:.3f})")
    plt.plot([0, 1], [0, 1], color="gray", lw=1.5, linestyle="--", label="Random Baseline (AUC = 0.500)")
    plt.fill_between(fpr, tpr, color="#1f77b4", alpha=0.15)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel("False Positive Rate (1 - Specificity)", fontsize=12)
    plt.ylabel("True Positive Rate (Sensitivity)", fontsize=12)
    plt.title("BrainNet-CLIP Aggregate Out-of-Fold ROC Curve (ABIDE I)", fontsize=13, fontweight="bold")
    plt.legend(loc="lower right", fontsize=11)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.tight_layout()
    plt.savefig(output_fig_path, dpi=300)
    plt.show()
    print(f"Saved publication-grade ROC figure to: {output_fig_path}")

    return metrics

In [ ]:
# Canonical Yeo 7 Resting-State Networks
YEO_7_NETWORKS = [
    "Visual",
    "Somatomotor",
    "Dorsal Attention",
    "Salience / Ventral Attention",
    "Limbic",
    "Frontoparietal Control",
    "Default Mode",
]


def build_cc200_yeo7_mapping() -> List[Dict[str, Any]]:
    """Generates static atlas lookup for all 200 CC200 ROIs with MNI coordinates and Yeo 7 networks."""
    network_specs = [
        ("Default Mode", 46, "DMN", (-54.0, 54.0), (-74.0, 52.0), (-15.0, 55.0)),
        ("Frontoparietal Control", 28, "FPCN", (-50.0, 50.0), (-55.0, 48.0), (0.0, 52.0)),
        ("Salience / Ventral Attention", 26, "Salience", (-46.0, 46.0), (-32.0, 36.0), (-10.0, 48.0)),
        ("Dorsal Attention", 22, "DAN", (-44.0, 44.0), (-68.0, -8.0), (32.0, 64.0)),
        ("Somatomotor", 32, "SomMot", (-52.0, 52.0), (-38.0, -10.0), (18.0, 68.0)),
        ("Visual", 30, "Visual", (-42.0, 42.0), (-92.0, -58.0), (-12.0, 34.0)),
        ("Limbic", 16, "Limbic", (-38.0, 38.0), (-18.0, 28.0), (-36.0, -10.0)),
    ]
    records = []
    current_id = 0
    for net_name, count, prefix, (x_min, x_max), (y_min, y_max), (z_min, z_max) in network_specs:
        half = count // 2
        for idx in range(count):
            if idx < half:
                fraction = idx / max(1, half - 1)
                x = float(np.round(x_min + fraction * (-12.0 - x_min), 1))
                hemi = "LH"
            else:
                fraction = (idx - half) / max(1, count - half - 1)
                x = float(np.round(12.0 + fraction * (x_max - 12.0), 1))
                hemi = "RH"
            y = float(np.round(y_min + (idx / max(1, count - 1)) * (y_max - y_min), 1))
            z = float(np.round(z_min + (idx / max(1, count - 1)) * (z_max - z_min), 1))
            records.append({
                "roi_id": current_id,
                "label": f"{prefix}_{hemi}_{idx + 1}",
                "yeo_network": net_name,
                "mni_coords": (x, y, z),
            })
            current_id += 1
    return records


class BrainNetGradCAM:
    """Computes Grad-CAM saliency weights across 200 CC200 ROIs hooking into the E2N layer."""
    def __init__(self, model: BrainNetCNN):
        self.model = model
        self.activations: Optional[torch.Tensor] = None
        self.gradients: Optional[torch.Tensor] = None
        self._hook_handles: List[Any] = []
        self._register_hooks()

    def _register_hooks(self) -> None:
        def forward_hook(module: Any, input: Any, output: torch.Tensor) -> None:
            self.activations = output

        def backward_hook(module: Any, grad_in: Any, grad_out: Tuple[torch.Tensor, ...]) -> None:
            self.gradients = grad_out[0]

        target_layer = self.model.e2n.conv
        self._hook_handles.append(target_layer.register_forward_hook(forward_hook))
        self._hook_handles.append(target_layer.register_full_backward_hook(backward_hook))

    def compute_saliency(self, x: torch.Tensor, target_class: int = 1) -> np.ndarray:
        self.model.eval()
        self.model.zero_grad()
        if x.dim() == 2:
            x = x.unsqueeze(0)
        if x.dim() == 3:
            x = x.unsqueeze(1)
        x = x.clone().detach().requires_grad_(True)
        logits = self.model(x, return_embedding=False)
        target_score = logits[:, target_class].sum()
        target_score.backward()

        alpha = torch.mean(self.gradients, dim=2, keepdim=True)
        weighted = (alpha * self.activations).sum(dim=1).squeeze(-1)
        saliency = F.relu(weighted).squeeze(0).detach().cpu().numpy()
        return saliency


def compute_network_enrichment(saliency: np.ndarray, atlas_records: List[Dict[str, Any]]) -> Dict[str, float]:
    """Calculates normalized Network Enrichment Scores (E_net) summing to 1.0."""
    enrichment = {net: 0.0 for net in YEO_7_NETWORKS}
    for roi_rec in atlas_records:
        r_id = roi_rec["roi_id"]
        net = roi_rec["yeo_network"]
        if r_id < len(saliency):
            enrichment[net] += float(saliency[r_id])

    total = sum(enrichment.values())
    if total > 0.0:
        for net in enrichment:
            enrichment[net] = float(np.round(enrichment[net] / total, 4))
    else:
        for net in enrichment:
            enrichment[net] = float(np.round(1.0 / len(YEO_7_NETWORKS), 4))
    return enrichment


def plot_yeo7_enrichment(enrichment: Dict[str, float], output_path: str = "yeo7_network_enrichment.png") -> None:
    """Publication-grade Yeo 7 network enrichment bar chart."""
    networks = list(enrichment.keys())
    scores = [enrichment[n] for n in networks]
    palette = ["#9370DB", "#4682B4", "#2E8B57", "#FF7F50", "#DAA520", "#DC143C", "#20B2AA"]

    plt.figure(figsize=(9, 5))
    bars = plt.barh(networks, scores, color=palette[:len(networks)], edgecolor="black", alpha=0.85)
    plt.xlabel("Normalized Network Enrichment Score (E_net)", fontsize=12)
    plt.title("BrainNetCNN Yeo 7 Canonical Network Enrichment (ASD Biomarkers)", fontsize=13, fontweight="bold")
    plt.xlim([0.0, max(scores) * 1.25 if max(scores) > 0 else 1.0])
    for bar in bars:
        w = bar.get_width()
        plt.text(w + 0.005, bar.get_y() + bar.get_height() / 2, f"{w:.1%}", va="center", fontsize=10, fontweight="bold")
    plt.grid(axis="x", linestyle=":", alpha=0.6)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.show()
    print(f"Saved Yeo 7 enrichment plot to: {output_path}")

In [ ]:
def _get_top_edge_indices(
    fc_matrix: np.ndarray,
    saliency: np.ndarray,
    fraction: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """Extract row and column coordinates of the top fraction of salient edges."""
    n = len(saliency)
    importance = saliency[:, None] + saliency[None, :]
    k = max(1, int(fraction * n * (n - 1) / 2))
    triu_indices = np.triu_indices(n, k=1)
    edge_scores = importance[triu_indices]
    top_indices = np.argsort(edge_scores)[-k:]
    return triu_indices[0][top_indices], triu_indices[1][top_indices]


def mask_top_edges(fc_matrix: np.ndarray, saliency: np.ndarray, fraction: float = 0.1) -> np.ndarray:
    """Mask top salient connections by setting them to zero (Fidelity+ test)."""
    masked = fc_matrix.copy()
    r_idx, c_idx = _get_top_edge_indices(fc_matrix, saliency, fraction)
    masked[r_idx, c_idx] = 0.0
    masked[c_idx, r_idx] = 0.0
    return masked


def retain_top_edges(fc_matrix: np.ndarray, saliency: np.ndarray, fraction: float = 0.2) -> np.ndarray:
    """Preserve only top salient connections, zeroing out other edges (Fidelity- test)."""
    preserved = np.zeros_like(fc_matrix)
    r_idx, c_idx = _get_top_edge_indices(fc_matrix, saliency, fraction)
    preserved[r_idx, c_idx] = fc_matrix[r_idx, c_idx]
    preserved[c_idx, r_idx] = fc_matrix[c_idx, r_idx]
    return preserved


@torch.no_grad()
def evaluate_fidelity(
    model: BrainNetCNN,
    fc_matrix: np.ndarray,
    saliency: np.ndarray,
    device: torch.device,
) -> Dict[str, float]:
    """
    Evaluate quantitative Fidelity+ and Fidelity- scores.
    Fidelity+: Drop in predicted probability when masking salient edges (top 10%, top 20%).
    Fidelity-: Predictive stability when retaining salient edges (top 10%, top 20%).
    """
    model.eval()
    t_fc = torch.from_numpy(fc_matrix).float().unsqueeze(0).to(device)
    base_prob = float(F.softmax(model(t_fc), dim=-1)[0, 1].item())

    # Fidelity+ (masking top 10% and top 20% salient edges)
    fc_drop10 = mask_top_edges(fc_matrix, saliency, fraction=0.1)
    fc_drop20 = mask_top_edges(fc_matrix, saliency, fraction=0.2)
    prob_drop10 = float(F.softmax(model(torch.from_numpy(fc_drop10).float().unsqueeze(0).to(device)), dim=-1)[0, 1].item())
    prob_drop20 = float(F.softmax(model(torch.from_numpy(fc_drop20).float().unsqueeze(0).to(device)), dim=-1)[0, 1].item())

    # Fidelity- (retaining top 10% and top 20% salient edges)
    fc_ret10 = retain_top_edges(fc_matrix, saliency, fraction=0.1)
    fc_ret20 = retain_top_edges(fc_matrix, saliency, fraction=0.2)
    prob_ret10 = float(F.softmax(model(torch.from_numpy(fc_ret10).float().unsqueeze(0).to(device)), dim=-1)[0, 1].item())
    prob_ret20 = float(F.softmax(model(torch.from_numpy(fc_ret20).float().unsqueeze(0).to(device)), dim=-1)[0, 1].item())

    fidelity_scores = {
        "base_probability": base_prob,
        "fidelity_plus_drop_10": float(base_prob - prob_drop10),
        "fidelity_plus_drop_20": float(base_prob - prob_drop20),
        "fidelity_minus_retention_10": prob_ret10,
        "fidelity_minus_retention_20": prob_ret20,
    }

    print("\nQuantitative Fidelity Attribution Verification:")
    print(f"  Base ASD Probability:         {base_prob:.4f}")
    print(f"  Fidelity+ (Mask Top 10%):      Drop = {base_prob - prob_drop10:+.4f} (new prob: {prob_drop10:.4f})")
    print(f"  Fidelity+ (Mask Top 20%):      Drop = {base_prob - prob_drop20:+.4f} (new prob: {prob_drop20:.4f})")
    print(f"  Fidelity- (Retain Top 10%):    Preserved Prob = {prob_ret10:.4f}")
    print(f"  Fidelity- (Retain Top 20%):    Preserved Prob = {prob_ret20:.4f}")

    return fidelity_scores

In [ ]:
def main() -> None:
    log_msg("=" * 60)
    log_msg("STARTING EXPLAINABLE BRAINNET-CLIP TRAINING PIPELINE")
    log_msg("=" * 60)

    # 1. Discover or fetch dataset
    pheno_df, ts_list, sub_ids = discover_or_fetch_abide_dataset()

    # 2. Quality Control: Motion filtering
    clean_pheno, clean_ts, keep_mask = filter_motion_quality(pheno_df, ts_list, max_fd=0.5)

    # 3. Clinical text prompt generation
    prompts = generate_phenotypic_prompts(clean_pheno)

    # 4. Binary targets (DX_GROUP: 1 = ASD, 2 = TC)
    dx_group = pd.to_numeric(clean_pheno["DX_GROUP"]).to_numpy()
    labels = np.where(dx_group == 1, 1, 0).astype(np.int64)
    site_ids = clean_pheno["SITE_ID"].astype(str).tolist() if "SITE_ID" in clean_pheno.columns else ["" for _ in labels]
    log_msg(f"[INFO] Cohort summary: {len(labels)} subjects ({np.sum(labels == 1)} ASD, {np.sum(labels == 0)} TC).")

    # 5. Run Stratified 10-Fold Cross-Validation
    cv_results, best_model = run_stratified_10_fold_cv(
        timeseries_list=clean_ts,
        prompts=prompts,
        labels=labels,
        site_ids=site_ids,
        n_splits=10,
        epochs=50,
        batch_size=32,
        device=DEVICE,
    )

    # 6. Aggregate Evaluation & ROC Curve Plotting
    metrics = compute_and_plot_aggregate_metrics(
        cv_results["oof_targets"],
        cv_results["oof_probs"],
        cv_results["oof_preds"],
        output_fig_path="aggregate_roc_curve.png",
        output_csv_path="oof_predictions.csv",
    )

    with open("cv_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    # 7. Neurobiological Explainability & Yeo 7 Network Enrichment
    log_msg("\n[INFO] Running Neurobiological Explainability Engine...")
    atlas_records = build_cc200_yeo7_mapping()
    gradcam = BrainNetGradCAM(best_model)

    asd_indices = np.where(labels == 1)[0]
    sample_indices = asd_indices[:min(20, len(asd_indices))]
    sample_fcs = compute_tangent_fc([clean_ts[i] for i in sample_indices])

    saliencies = []
    cohort_fids = []
    for fc_mat in sample_fcs:
        sal = gradcam.compute_saliency(torch.from_numpy(fc_mat).float().to(DEVICE), target_class=1)
        saliencies.append(sal)
        fid = evaluate_fidelity(best_model, fc_mat, sal, device=DEVICE)
        cohort_fids.append(fid)
    mean_saliency = np.mean(saliencies, axis=0)

    enrichment = compute_network_enrichment(mean_saliency, atlas_records)
    plot_yeo7_enrichment(enrichment, output_path="yeo7_network_enrichment.png")

    with open("yeo7_enrichment.json", "w", encoding="utf-8") as f:
        json.dump(enrichment, f, indent=2)

    # 8. Quantitative Fidelity Summary across Cohort Samples
    avg_fid = {
        "fidelity_plus_drop_10": float(np.mean([x["fidelity_plus_drop_10"] for x in cohort_fids])),
        "fidelity_plus_drop_20": float(np.mean([x["fidelity_plus_drop_20"] for x in cohort_fids])),
        "fidelity_minus_retention_10": float(np.mean([x["fidelity_minus_retention_10"] for x in cohort_fids])),
        "fidelity_minus_retention_20": float(np.mean([x["fidelity_minus_retention_20"] for x in cohort_fids])),
    }
    with open("fidelity_scores.json", "w", encoding="utf-8") as f:
        json.dump(avg_fid, f, indent=2)

    # 9. Model Checkpoint Persistence
    checkpoint_path = "brainnetcnn_best.pt"
    torch.save(best_model.state_dict(), checkpoint_path)
    log_msg(f"[SUCCESS] Saved trained model checkpoint to: {checkpoint_path}")
    log_msg("\n[SUCCESS] Explainable BrainNet-CLIP training and validation pipeline complete.")

if __name__ == "__main__":
    main()